<a href="https://colab.research.google.com/github/Nihat0311/NLP/blob/main/Recurrent_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
from collections import Counter
import re
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn

In [ ]:
dataset=load_dataset('glue','sst2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [ ]:
print(dataset['train'][0])

{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [ ]:
dataset['train'].features

{'sentence': Value('string'),
 'label': ClassLabel(names=['negative', 'positive']),
 'idx': Value('int32')}

In [ ]:
train_data=dataset['train'].shuffle(seed=42)
valid_data=dataset['validation'].shuffle(seed=42)

In [ ]:
train_data

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 67349
})

In [ ]:
print('Train labels', Counter(train_data['label']))
print('Valid labels', Counter(valid_data['label']))

Train labels Counter({1: 37569, 0: 29780})
Valid labels Counter({1: 444, 0: 428})


In [ ]:
def tokenize(text):
  return re.findall(r"\b\w+\b|[!?.,]",text.lower())

In [ ]:
counter=Counter()

for item in train_data:
  tokens=tokenize(item['sentence'])
  counter.update(tokens)

vocab={
    "<PAD>":0,
    "<UNK>":1
}

min_freq=2

for word,count in counter.items():
  if count>=min_freq:
    vocab[word]=len(vocab)

print(len(vocab))
print(list(vocab.items())[:20])

13373
[('<PAD>', 0), ('<UNK>', 1), ('klein', 2), (',', 3), ('charming', 4), ('in', 5), ('comedies', 6), ('like', 7), ('american', 8), ('pie', 9), ('and', 10), ('dead', 11), ('on', 12), ('election', 13), ('be', 14), ('fruitful', 15), ('soulful', 16), ('the', 17), ('proud', 18), ('warrior', 19)]


In [ ]:
max_length=30

def encode_text(text,vocab):
  tokens=tokenize(text)
  ids=[vocab.get(token,vocab["<UNK>"]) for token in tokens]
  return ids

In [ ]:
def pad_or_truncate(ids,max_length):
  if len(ids)>max_length:
    ids=ids[:max_length]

  return ids + [vocab["<PAD>"]]*(max_length-len(ids))

In [ ]:
class SentimentDataset(Dataset):
  def __init__(self,hf_dataset,vocab,max_length):
    self.dataset=hf_dataset
    self.vocab=vocab
    self.max_length=max_length

  def __len__(self):
    return len(self.dataset)
  def __getitem__(self,idx):
    text=self.dataset[idx]['sentence']
    label=self.dataset[idx]['label']

    ids=encode_text(text,self.vocab)

    length=min(len(ids),self.max_length)

    ids=pad_or_truncate(ids,self.max_length)

    X=torch.tensor(ids,dtype=torch.long)
    y=torch.tensor(label,dtype=torch.long)
    length=torch.tensor(length,dtype=torch.long)

    return X,y,length

In [ ]:
train_set=SentimentDataset(train_data,vocab,max_length)
valid_set=SentimentDataset(valid_data,vocab,max_length)

train_loader=DataLoader(train_set,batch_size=64,shuffle=True)
valid_loader=DataLoader(valid_set,batch_size=64)

X_batch,y_batch,lengths_batch=next(iter(train_loader))
print(X_batch.shape)
print(y_batch.shape)
print(lengths_batch.shape)
print(X_batch[0])
print(y_batch[0])
print(lengths_batch[0])

torch.Size([64, 30])
torch.Size([64])
torch.Size([64])
tensor([ 221,  519,   10, 2851,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0])
tensor(0)
tensor(4)


In [ ]:
from torch.func import functional_call
class SimpleRNNClassifier(nn.Module):
    def __init__(self, hidden_size, vocab_size,embedding_dim,num_classes):
      super().__init__()

      self.embedding=nn.Embedding(
          num_embeddings=vocab_size,
          embedding_dim=embedding_dim,
          padding_idx=0
          )
      self.rnn=nn.RNN(
          input_size=embedding_dim,
          hidden_size=hidden_size,
          batch_first=True,
          )
      self.fc=nn.Linear(
          in_features=hidden_size,
          out_features=num_classes
          )
    def forward(self,X,lengths):
      embedded=self.embedding(X)
      outputs,last_hidden=self.rnn(embedded)
      batch_size=outputs.size(0)

      final_hidden=outputs[
          torch.arange(batch_size,device=outputs.device),
          lengths-1
      ]

      logits=self.fc(final_hidden)

      return logits

In [ ]:
vocab_size=len(vocab)
embedding_dim=64
hidden_size=128
num_classes=2

device='cuda' if torch.cuda.is_available() else 'cpu'

model=SimpleRNNClassifier(
    hidden_size=hidden_size,
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_classes=num_classes
).to(device)

X_batch,y_batch,lengths_batch=next(iter(train_loader))
X_batch=X_batch.to(device)
y_batch=y_batch.to(device)
lengths_batch=lengths_batch.to(device)

embedded=model.embedding(X_batch)
outputs,last_hidden=model.rnn(embedded)
batch_size=outputs.size(0)
final_hidden=outputs[
    torch.arange(batch_size,device=outputs.device),
    lengths_batch-1
]
logits=model.fc(final_hidden)

print(embedded.shape)
print(outputs.shape)
print(last_hidden.shape)

torch.Size([64, 30, 64])
torch.Size([64, 30, 128])
torch.Size([1, 64, 128])


In [ ]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
def train_one_epoch(model,loader,loss_fn,optimizer,device):
  model.train()

  total_loss=0
  correct=0
  total=0
  for X_batch,y_batch,lengths in loader:
    X_batch=X_batch.to(device)
    y_batch=y_batch.to(device)
    lengths=lengths.to(device)

    logits=model(X_batch,lengths)
    loss=loss_fn(logits,y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()* X_batch.size(0)

    preds=torch.argmax(logits,dim=1)
    correct+=(preds==y_batch).sum().item()
    total+=y_batch.size(0)

  return total_loss/total,correct/total


In [ ]:
def evaluate(model,loader,loss_fn,device):
  model.eval()
  total_loss=0
  correct=0
  total=0
  with torch.no_grad():
    for X_batch,y_batch,lengths in loader:
      X_batch=X_batch.to(device)
      y_batch=y_batch.to(device)
      lengths=lengths.to(device)

      logits=model(X_batch,lengths)
      loss=loss_fn(logits,y_batch)

      total_loss+= loss.item() * X_batch.size(0)

      preds=torch.argmax(logits,dim=1)
      correct+=(preds==y_batch).sum().item()
      total+=y_batch.size(0)

  return total_loss/total,correct/total

In [ ]:
n_epochs=10
for epoch in range(n_epochs):
  train_loss,train_acc=train_one_epoch(model,train_loader,loss_fn,optimizer,device)
  valid_loss,valid_acc=evaluate(model,valid_loader,loss_fn,device)

  print(
      f"Epoch: {epoch+1:02d} | ",
      f"Train Loss: {train_loss:.4f}",
      f"Train Acc: {train_acc:.4f}",
      f"Valid Loss: {valid_loss:.4f}",
      f"Valid Acc: {valid_acc:.4f}"
  )

Epoch: 01 |  Train Loss: 0.5606 Train Acc: 0.7037 Valid Loss: 0.5733 Valid Acc: 0.7167
Epoch: 02 |  Train Loss: 0.3438 Train Acc: 0.8568 Valid Loss: 0.5269 Valid Acc: 0.7856
Epoch: 03 |  Train Loss: 0.2495 Train Acc: 0.9028 Valid Loss: 0.5374 Valid Acc: 0.7993
Epoch: 04 |  Train Loss: 0.1976 Train Acc: 0.9265 Valid Loss: 0.5615 Valid Acc: 0.8039
Epoch: 05 |  Train Loss: 0.1658 Train Acc: 0.9396 Valid Loss: 0.6264 Valid Acc: 0.8028
Epoch: 06 |  Train Loss: 0.1400 Train Acc: 0.9492 Valid Loss: 0.6467 Valid Acc: 0.8085
Epoch: 07 |  Train Loss: 0.1184 Train Acc: 0.9580 Valid Loss: 0.6721 Valid Acc: 0.8016
Epoch: 08 |  Train Loss: 0.1022 Train Acc: 0.9634 Valid Loss: 0.6489 Valid Acc: 0.8222
Epoch: 09 |  Train Loss: 0.0873 Train Acc: 0.9684 Valid Loss: 0.6649 Valid Acc: 0.8165
Epoch: 10 |  Train Loss: 0.0771 Train Acc: 0.9727 Valid Loss: 0.7958 Valid Acc: 0.8142


In [ ]:
def predict_sentiment(text,model,vocab,max_length,device):
  model.eval()
  ids=encode_text(text,vocab)
  length=min(len(ids),max_length)
  ids=pad_or_truncate(ids,max_length)

  X=torch.tensor([ids],dtype=torch.long).to(device)
  lengths=torch.tensor([length],dtype=torch.long).to(device)

  with torch.no_grad():
    logits=model(X,lengths)
    probs=torch.softmax(logits,dim=1)
    pred=torch.argmax(probs,dim=1).item()

  label='positive' if pred==1 else 'negative'

  return label , probs.cpu()

In [ ]:
sentences = [
    'i love the boringness of the movie very much its the most disgusting movie i have ever seen'
]

for sentence in sentences:
    label, probs = predict_sentiment(
        sentence,
        model,
        vocab,
        max_length,
        device
    )

    print(sentence, "→", label, probs)

i love the boringness of the movie very much its the most disgusting movie i have ever seen → positive tensor([[0.0044, 0.9956]])


LSTM

In [ ]:
class SimpleLSTMClassifier(nn.Module):
    def __init__(self, hidden_size, vocab_size,embedding_dim,num_classes):
      super().__init__()

      self.embedding=nn.Embedding(
          num_embeddings=vocab_size,
          embedding_dim=embedding_dim,
          padding_idx=0
          )
      self.lstm=nn.LSTM(
          input_size=embedding_dim,
          hidden_size=hidden_size,
          batch_first=True
          )
      self.fc=nn.Linear(
          in_features=hidden_size,
          out_features=num_classes
          )
    def forward(self,X,lengths):
      embedded=self.embedding(X)

      outputs,(last_hidden,last_cell)=self.lstm(embedded)

      batch_size=outputs.size(0)

      final_hidden=outputs[
          torch.arange(batch_size,device=outputs.device),
          lengths-1
      ]

      logits=self.fc(final_hidden)

      return logits

In [ ]:
model=SimpleLSTMClassifier(
    hidden_size=hidden_size,
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_classes=num_classes
).to(device)

In [ ]:
embedded=model.embedding(X_batch)
outputs,(last_hidden,last_cell)=model.lstm(embedded)


In [ ]:
print(embedded.shape)
print(outputs.shape)
print(last_hidden.shape)
print(last_cell.shape)
print(X_batch.shape)

torch.Size([64, 30, 64])
torch.Size([64, 30, 128])
torch.Size([1, 64, 128])
torch.Size([1, 64, 128])
torch.Size([64, 30])


In [ ]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
num_epochs=10

for epoch in range(num_epochs):
  train_loss,train_acc=train_one_epoch(model,train_loader,loss_fn,optimizer,device)
  valid_loss,valid_acc=evaluate(model,valid_loader,loss_fn,device)
  print(
      f"Epoch: {epoch+1:02d} | ",
      f"Train Loss: {train_loss:.4f}",
      f"Train Acc: {train_acc:.4f}",
      f"Valid Loss: {valid_loss:.4f}",
      f"Valid Acc: {valid_acc:.4f}"
  )

Epoch: 01 |  Train Loss: 0.5064 Train Acc: 0.7405 Valid Loss: 0.4989 Valid Acc: 0.7821
Epoch: 02 |  Train Loss: 0.2758 Train Acc: 0.8860 Valid Loss: 0.5731 Valid Acc: 0.8050
Epoch: 03 |  Train Loss: 0.1848 Train Acc: 0.9282 Valid Loss: 0.6312 Valid Acc: 0.7810
Epoch: 04 |  Train Loss: 0.1349 Train Acc: 0.9498 Valid Loss: 0.6795 Valid Acc: 0.8016
Epoch: 05 |  Train Loss: 0.1020 Train Acc: 0.9625 Valid Loss: 0.6539 Valid Acc: 0.8303
Epoch: 06 |  Train Loss: 0.0806 Train Acc: 0.9706 Valid Loss: 0.7218 Valid Acc: 0.8039
Epoch: 07 |  Train Loss: 0.0633 Train Acc: 0.9765 Valid Loss: 0.7010 Valid Acc: 0.8005
Epoch: 08 |  Train Loss: 0.0506 Train Acc: 0.9812 Valid Loss: 0.8908 Valid Acc: 0.8050
Epoch: 09 |  Train Loss: 0.0426 Train Acc: 0.9848 Valid Loss: 0.8547 Valid Acc: 0.8154
Epoch: 10 |  Train Loss: 0.0356 Train Acc: 0.9873 Valid Loss: 0.9046 Valid Acc: 0.8211


In [ ]:
sentences = [
    "this movie was not very good but the ending was surprisingly emotional and beautiful",
    "i thought the film would be boring at first but it slowly became one of the most touching stories i have seen",
    "the acting was decent and the music was nice but the story was too predictable and the ending felt weak",
    "although the movie started slowly the characters became more interesting and the final scenes were really powerful",
    "i wanted to enjoy this film because the trailer looked amazing but the actual story was confusing and disappointing",
    "the first half was funny and charming but after a while the movie became repetitive and lost its energy",
    "this was not a perfect movie but it had enough heart and emotion to make me enjoy it",
    "i did not expect much from this film but it turned out to be clever warm and genuinely entertaining",
    "the movie had beautiful visuals and strong performances but the plot was messy and hard to follow",
    "even though some scenes were too long the overall story was meaningful and the ending made the film worth watching",
    "the film looked expensive and stylish but it felt empty because the characters were poorly written",
    "i almost stopped watching because the beginning was slow but the second half completely changed my opinion",
    "this movie was not terrible but it was also not interesting enough to recommend",
    "the story was simple but the performances were honest and the emotional moments worked really well",
    "despite a few funny moments the movie felt lazy predictable and forgettable",
    "i loved how the film slowly built tension and delivered a satisfying ending",
    "the movie tries to be deep and emotional but most of the dialogue feels unnatural",
    "it was a long film but i never felt bored because the characters were so engaging",
    "the ending was powerful but the rest of the movie was too slow and unfocused",
    "i did not like the first thirty minutes but after that the movie became surprisingly good"
]

for sentence in sentences:
    label, probs = predict_sentiment(
        sentence,
        model,
        vocab,
        max_length,
        device
    )

    print(sentence, "→", label, probs)
for sentence in sentences:
  label,probs=predict_sentiment(
      sentence,model,vocab,max_length,device
  )
  print(sentence,label,probs)

this movie was not very good but the ending was surprisingly emotional and beautiful → positive tensor([[0.0611, 0.9389]])
i thought the film would be boring at first but it slowly became one of the most touching stories i have seen → positive tensor([[1.1629e-05, 9.9999e-01]])
the acting was decent and the music was nice but the story was too predictable and the ending felt weak → negative tensor([[9.9960e-01, 4.0202e-04]])
although the movie started slowly the characters became more interesting and the final scenes were really powerful → negative tensor([[0.5721, 0.4279]])
i wanted to enjoy this film because the trailer looked amazing but the actual story was confusing and disappointing → positive tensor([[0.1612, 0.8388]])
the first half was funny and charming but after a while the movie became repetitive and lost its energy → negative tensor([[9.9990e-01, 9.5165e-05]])
this was not a perfect movie but it had enough heart and emotion to make me enjoy it → positive tensor([[0.0157, 0